In [0]:
# Import the required Delta Lake, Python and PySpark components

from datetime import datetime

from delta.tables import DeltaTable

from pyspark.sql.window import Window

from pyspark.sql.functions import (
    col,
    count,
    countDistinct,
    current_timestamp,
    lit,
    regexp_replace,
    round as spark_round,
    row_number,
    trim,
    upper,
    when
)

In [0]:
# Receive pipeline parameters and define table names

dbutils.widgets.text(
    "batch_id",
    "2009-12",
    "Batch ID"
)

dbutils.widgets.text(
    "run_id",
    "manual-run-001",
    "Run ID"
)

batch_id = dbutils.widgets.get("batch_id")
run_id = dbutils.widgets.get("run_id")

bronze_table = "online_retail.bronze.transactions_raw"
silver_table = "online_retail.silver.transactions_clean"
control_table = "online_retail.control.pipeline_runs"
layer_name = "silver"

print(f"Run ID: {run_id}")
print(f"Batch ID: {batch_id}")
print(f"Bronze table: {bronze_table}")
print(f"Silver table: {silver_table}")

Run ID: silver-cleanup-test-001
Batch ID: 2010-02
Bronze table: online_retail.bronze.transactions_raw
Silver table: online_retail.silver.transactions_clean


In [0]:
# Validate that batch_id is a valid month in exact YYYY-MM format

try:
    batch_month = datetime.strptime(batch_id, "%Y-%m")

    if batch_month.strftime("%Y-%m") != batch_id:
        raise ValueError

except ValueError as error:
    raise ValueError(
        f"Invalid batch_id: {batch_id}. Expected YYYY-MM."
    ) from error

else:
    print(f"Valid batch ID: {batch_id}")

Valid batch ID: 2010-02


In [0]:
# Record that Silver processing has started

if not spark.catalog.tableExists(control_table):
    raise ValueError(
        f"Control table does not exist: {control_table}"
    )


started_audit_df = (
    spark.range(1)
    .select(
        lit(run_id).alias("run_id"),
        lit(batch_id).alias("batch_id"),
        lit(layer_name).alias("layer_name"),
        lit("STARTED").alias("status"),
        current_timestamp().alias("start_timestamp"),
        lit(None).cast("timestamp").alias("end_timestamp"),
        lit(None).cast("long").alias("input_row_count"),
        lit(None).cast("long").alias("output_row_count"),
        lit(None).cast("string").alias("error_message")
    )
)


control_delta_table = DeltaTable.forName(
    spark,
    control_table
)


(
    control_delta_table.alias("target")
    .merge(
        started_audit_df.alias("source"),
        """
        target.run_id = source.run_id
        AND target.batch_id = source.batch_id
        AND target.layer_name = source.layer_name
        """
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

print(
    f"Silver processing started for batch {batch_id}, "
    f"run {run_id}."
)

Silver processing started for batch 2010-02, run silver-cleanup-test-001.


In [0]:
# Read only the selected batch from the Bronze table

if not spark.catalog.tableExists(bronze_table):
    raise ValueError(
        f"Bronze table does not exist: {bronze_table}"
    )


bronze_batch_df = (
    spark.table(bronze_table)
    .filter(col("batch_id") == batch_id)
)

bronze_batch_row_count = bronze_batch_df.count()


if bronze_batch_row_count == 0:
    raise ValueError(
        f"Bronze contains no records for batch {batch_id}."
    )

print(
    f"Bronze batch {batch_id} contains "
    f"{bronze_batch_row_count:,} records."
)

Bronze batch 2010-02 contains 29,388 records.


In [0]:
# Profile important data-quality conditions before transformation

quality_profile_df = (
    bronze_batch_df
    .agg(
        count(
            when(col("invoice").isNull(), 1)
        ).alias("missing_invoice"),

        count(
            when(col("stock_code").isNull(), 1)
        ).alias("missing_stock_code"),

        count(
            when(col("description").isNull(), 1)
        ).alias("missing_description"),

        count(
            when(col("invoice_date").isNull(), 1)
        ).alias("missing_invoice_date"),

        count(
            when(col("customer_id").isNull(), 1)
        ).alias("missing_customer_id"),

        count(
            when(col("country").isNull(), 1)
        ).alias("missing_country"),

        count(
            when(col("quantity") < 0, 1)
        ).alias("negative_quantity"),

        count(
            when(col("quantity") == 0, 1)
        ).alias("zero_quantity"),

        count(
            when(col("price") < 0, 1)
        ).alias("negative_price"),

        count(
            when(col("price") == 0, 1)
        ).alias("zero_price"),

        count(
            when(
                upper(col("invoice")).startswith("C"),
                1
            )
        ).alias("cancelled_invoice")
    )
)

quality_profile_df.show()

+---------------+------------------+-------------------+--------------------+-------------------+---------------+-----------------+-------------+--------------+----------+-----------------+
|missing_invoice|missing_stock_code|missing_description|missing_invoice_date|missing_customer_id|missing_country|negative_quantity|zero_quantity|negative_price|zero_price|cancelled_invoice|
+---------------+------------------+-------------------+--------------------+-------------------+---------------+-----------------+-------------+--------------+----------+-----------------+
|              0|                 0|                510|                   0|               5482|              0|             1023|            0|             0|       531|              576|
+---------------+------------------+-------------------+--------------------+-------------------+---------------+-----------------+-------------+--------------+----------+-----------------+



In [0]:
# Remove duplicate business records deterministically

business_columns = [
    "invoice",
    "stock_code",
    "description",
    "quantity",
    "invoice_date",
    "price",
    "customer_id",
    "country"
]


deduplication_window = (
    Window
    .partitionBy(*business_columns)
    .orderBy(
        col("source_sheet"),
        col("source_row_number"),
        col("record_id")
    )
)


deduplicated_batch_df = (
    bronze_batch_df
    .withColumn(
        "duplicate_rank",
        row_number().over(deduplication_window)
    )
    .filter(col("duplicate_rank") == 1)
    .drop("duplicate_rank")
)


deduplicated_batch_row_count = (
    deduplicated_batch_df.count()
)

duplicate_rows_removed = (
    bronze_batch_row_count
    - deduplicated_batch_row_count
)


print(
    f"Rows before deduplication: "
    f"{bronze_batch_row_count:,}"
)

print(
    f"Rows after deduplication: "
    f"{deduplicated_batch_row_count:,}"
)

print(
    f"Duplicate rows removed: "
    f"{duplicate_rows_removed:,}"
)

Rows before deduplication: 29,388
Rows after deduplication: 29,058
Duplicate rows removed: 330


In [0]:
# Trim text fields and standardize Customer IDs

cleaned_batch_df = (
    deduplicated_batch_df
    .withColumn(
        "invoice",
        trim(col("invoice"))
    )
    .withColumn(
        "stock_code",
        trim(col("stock_code"))
    )
    .withColumn(
        "description",
        trim(col("description"))
    )
    .withColumn(
        "customer_id",
        regexp_replace(
            trim(col("customer_id")),
            r"\.0$",
            ""
        )
    )
    .withColumn(
        "country",
        trim(col("country"))
    )
)


cleaned_batch_row_count = cleaned_batch_df.count()


if (
    cleaned_batch_row_count
    != deduplicated_batch_row_count
):
    raise ValueError(
        "Cleaning unexpectedly changed the batch row count."
    )

print(
    "Cleaned row count matches deduplicated row count: "
    f"{cleaned_batch_row_count == deduplicated_batch_row_count}"
)

Cleaned row count matches deduplicated row count: True


In [0]:
# Add reusable quality flags and calculate the transaction line total

silver_batch_df = (
    cleaned_batch_df
    .withColumn(
        "is_cancelled",
        when(
            upper(col("invoice")).startswith("C"),
            True
        ).otherwise(False)
    )
    .withColumn(
        "has_customer_id",
        when(
            col("customer_id").isNotNull()
            & (col("customer_id") != ""),
            True
        ).otherwise(False)
    )
    .withColumn(
        "has_description",
        when(
            col("description").isNotNull()
            & (col("description") != ""),
            True
        ).otherwise(False)
    )
    .withColumn(
        "is_positive_sale",
        when(
            (~col("is_cancelled"))
            & (col("quantity") > 0)
            & (col("price") > 0),
            True
        ).otherwise(False)
    )
    .withColumn(
        "line_total",
        spark_round(
            col("quantity") * col("price"),
            2
        )
    )
)

In [0]:
# Validate the prepared Silver batch before writing

silver_profile = (
    silver_batch_df
    .agg(
        count("*").alias("silver_batch_row_count"),

        countDistinct("record_id").alias(
            "distinct_record_id_count"
        ),

        count(
            when(col("record_id").isNull(), 1)
        ).alias("null_record_id_count"),

        count(
            when(col("is_cancelled"), 1)
        ).alias("cancelled_record_count"),

        count(
            when(col("is_positive_sale"), 1)
        ).alias("positive_sale_record_count"),

        count(
            when(~col("has_customer_id"), 1)
        ).alias("missing_customer_id_count"),

        count(
            when(~col("has_description"), 1)
        ).alias("missing_description_count")
    )
    .first()
)


silver_batch_row_count = (
    silver_profile["silver_batch_row_count"]
)

distinct_record_id_count = (
    silver_profile["distinct_record_id_count"]
)

null_record_id_count = (
    silver_profile["null_record_id_count"]
)

duplicate_record_id_count = (
    silver_batch_row_count
    - distinct_record_id_count
)


print(f"Silver batch rows: {silver_batch_row_count:,}")
print(f"Distinct record IDs: {distinct_record_id_count:,}")
print(f"Duplicate record IDs: {duplicate_record_id_count:,}")
print(f"Null record IDs: {null_record_id_count:,}")
print(
    "Cancelled records: "
    f"{silver_profile['cancelled_record_count']:,}"
)
print(
    "Positive-sale records: "
    f"{silver_profile['positive_sale_record_count']:,}"
)
print(
    "Missing Customer IDs: "
    f"{silver_profile['missing_customer_id_count']:,}"
)
print(
    "Missing descriptions: "
    f"{silver_profile['missing_description_count']:,}"
)


if silver_batch_row_count == 0:
    raise ValueError(
        f"Prepared Silver batch {batch_id} is empty."
    )

if (
    silver_batch_row_count
    != deduplicated_batch_row_count
):
    raise ValueError(
        "Silver row count does not match the "
        "deduplicated row count."
    )

if null_record_id_count > 0:
    raise ValueError(
        f"Found {null_record_id_count} null record IDs."
    )

if duplicate_record_id_count > 0:
    raise ValueError(
        f"Found {duplicate_record_id_count} duplicate "
        f"record IDs in the prepared Silver batch."
    )

print(f"Silver batch {batch_id} passed validation.")

Silver batch rows: 29,058
Distinct record IDs: 29,058
Duplicate record IDs: 0
Null record IDs: 0
Cancelled records: 575
Positive-sale records: 27,952
Missing Customer IDs: 5,480
Missing descriptions: 510
Silver batch 2010-02 passed validation.


In [0]:
# Upsert the prepared batch into the Silver Delta table

if spark.catalog.tableExists(silver_table):
    silver_delta_table = DeltaTable.forName(
        spark,
        silver_table
    )

    (
        silver_delta_table.alias("target")
        .merge(
            silver_batch_df.alias("source"),
            "target.record_id = source.record_id"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

    print(
        f"Merged batch {batch_id} into {silver_table}."
    )

else:
    (
        silver_batch_df
        .write
        .format("delta")
        .saveAsTable(silver_table)
    )

    print(
        f"Created {silver_table} with batch {batch_id}."
    )

Merged batch 2010-02 into online_retail.silver.transactions_clean.


In [0]:
# Verify the stored Silver batch after the Delta MERGE

stored_silver_batch_df = (
    spark.table(silver_table)
    .filter(col("batch_id") == batch_id)
)


stored_silver_profile = (
    stored_silver_batch_df
    .agg(
        count("*").alias("stored_batch_row_count"),

        countDistinct("record_id").alias(
            "stored_distinct_record_id_count"
        )
    )
    .first()
)


stored_batch_row_count = (
    stored_silver_profile["stored_batch_row_count"]
)

stored_distinct_record_id_count = (
    stored_silver_profile[
        "stored_distinct_record_id_count"
    ]
)

stored_duplicate_record_id_count = (
    stored_batch_row_count
    - stored_distinct_record_id_count
)


prepared_records_missing_from_silver = (
    silver_batch_df
    .select("record_id")
    .join(
        stored_silver_batch_df.select("record_id"),
        on="record_id",
        how="left_anti"
    )
    .count()
)


obsolete_stored_record_count = (
    stored_silver_batch_df
    .select("record_id")
    .join(
        silver_batch_df.select("record_id"),
        on="record_id",
        how="left_anti"
    )
    .count()
)


print(f"Stored Silver batch rows: {stored_batch_row_count:,}")
print(
    "Stored distinct record IDs: "
    f"{stored_distinct_record_id_count:,}"
)
print(
    "Stored duplicate record IDs: "
    f"{stored_duplicate_record_id_count:,}"
)
print(
    "Prepared records missing from Silver: "
    f"{prepared_records_missing_from_silver:,}"
)
print(
    "Obsolete stored Silver records: "
    f"{obsolete_stored_record_count:,}"
)


if stored_batch_row_count != silver_batch_row_count:
    raise ValueError(
        f"Stored Silver row count for batch {batch_id} "
        f"does not match the prepared row count."
    )

if stored_duplicate_record_id_count > 0:
    raise ValueError(
        f"Duplicate record IDs exist in stored Silver "
        f"batch {batch_id}."
    )

if prepared_records_missing_from_silver > 0:
    raise ValueError(
        f"{prepared_records_missing_from_silver} prepared "
        f"records are missing from Silver."
    )

if obsolete_stored_record_count > 0:
    raise ValueError(
        f"Stored Silver contains "
        f"{obsolete_stored_record_count} records that are "
        f"not present in the prepared batch."
    )

print(
    f"Stored Silver batch {batch_id} passed validation."
)

Stored Silver batch rows: 29,058
Stored distinct record IDs: 29,058
Stored duplicate record IDs: 0
Prepared records missing from Silver: 0
Obsolete stored Silver records: 0
Stored Silver batch 2010-02 passed validation.


In [0]:
# Mark Silver processing as successful in the control table

control_delta_table.update(
    condition=(
        (col("run_id") == run_id)
        & (col("batch_id") == batch_id)
        & (col("layer_name") == layer_name)
    ),
    set={
        "status": lit("SUCCESS"),
        "end_timestamp": current_timestamp(),
        "input_row_count": lit(bronze_batch_row_count),
        "output_row_count": lit(stored_batch_row_count),
        "error_message": lit(None).cast("string")
    }
)

print(
    f"Audit completed successfully for Silver, "
    f"batch {batch_id}, run {run_id}."
)

Audit completed successfully for Silver, batch 2010-02, run silver-cleanup-test-001.
